# DataHek OSS — 07 · Semantic layer

Define what words mean once: `error_count = count(*) where status = 'error'`.
The planner receives these definitions and prefers them over guessing.

In [1]:
import sys, os, tempfile
for _c in (os.getcwd(), os.path.join(os.getcwd(), "notebooks"), os.path.join(os.path.dirname(os.getcwd()), "notebooks")):
    if os.path.exists(os.path.join(_c, "datahek_demo.py")):
        sys.path.insert(0, _c)
        break
os.environ["DATAHEK_DB_PATH"] = os.path.join(tempfile.gettempdir(), "datahek_nb07.db")

from datahek_demo import build_demo_db, make_client

db = build_demo_db()
client, container = make_client(db)

## Define a metric

In [2]:
r = client.post("/metrics", json={
    "name": "error_count",
    "table": "traces",
    "aggregate": "count",
    "column": "*",
    "filter": "status = 'error'",
    "description": "number of error traces, failures",
})
print("created:", r.status_code)
print(client.get("/metrics").json())

created: 201
[{'id': 'ent_000001a09ffaad7829110048407c4fd885aa11d5', 'name': 'error_count', 'table': 'traces', 'aggregate': 'count', 'column': '*', 'filter': "status = 'error'", 'description': 'number of error traces, failures', 'created_at': '2026-09-14T12:53:20.887774+00:00'}, {'id': 'ent_000001a09ff7c1d2cac02632454f4fb99a3e7732', 'name': 'error_count', 'table': 'traces', 'aggregate': 'count', 'column': '*', 'filter': "status = 'error'", 'description': 'number of error traces, failures', 'created_at': '2026-09-14T12:50:09.490946+00:00'}]


## What the planner sees
The semantic catalog is injected into the planner prompt (relevance-ordered).

In [3]:
from datahek.engine.planner import _format_metrics

print(_format_metrics(client.get("/metrics").json(), "what is the error count?"))

Metric definitions (semantic layer — prefer these when relevant, keep the alias):
- error_count: count(*) WHERE status = 'error' on traces — "number of error traces, failures"
- error_count: count(*) WHERE status = 'error' on traces — "number of error traces, failures"


## Ask using the business term

In [4]:
r = client.post("/ask", json={"question": "What is the error count?",
                              "connection_id": "conn_demo"})
print("answer:", r.json()["answer"])
print("rows:", r.json()["rows"])

answer: Stub explanation: the result is shown in the table below.
rows: [{'service': 'payment-api', 'error_count': 1}, {'service': 'order-service', 'error_count': 1}, {'service': 'auth-service', 'error_count': 1}]


## Invalid definitions are refused
Aggregates are validated against the supported function set.

In [5]:
r = client.post("/metrics", json={"name": "median_ms", "table": "traces", "aggregate": "median"})
print("status:", r.status_code, "|", r.json()["message"])

status: 422 | Aggregate 'median' is not supported


**Takeaway:** metrics turn tribal knowledge into a catalog — consistent
across users, auditable, and validated.